# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [22]:
from huggingface_hub import notebook_login

notebook_login()

In [23]:
from huggingface_hub import get_token

token = get_token()

print("Token available:", token is not None)
print("Token prefix:", token[:5] + "..." if token else "None")

Token available: True
Token prefix: hf_Td...


In [24]:
from huggingface_hub import HfApi, get_token

token = get_token()

print("Token available:", token is not None)

api = HfApi(token=token)

try:
    info = api.repo_info(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset"
    )
    print("Dataset access: SUCCESS")
    print("Dataset:", info.id)

except Exception as e:
    print("Dataset access: FAILED")
    print(type(e).__name__)
    print(str(e))

Token available: True
Dataset access: SUCCESS
Dataset: FlyRank/internship-warehouse


In [25]:
import os
from huggingface_hub import get_token

token = get_token()

if token is None:
    raise RuntimeError("Hugging Face token not found.")

os.environ["HF_TOKEN"] = token

print("HF_TOKEN configured for this Colab session.")

HF_TOKEN configured for this Colab session.


In [26]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    PROVIDER credential_chain
)
""")

print("DuckDB Hugging Face secret created.")

DuckDB Hugging Face secret created.


In [27]:
test = con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
LIMIT 5
""").df()

print("HF → DuckDB access: SUCCESS")
test

HF → DuckDB access: SUCCESS


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [28]:
feature_frame = con.sql("""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions ELSE 0 END) AS feb_gsc_impressions,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS feb_gsc_clicks,

        AVG(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_avg_position ELSE NULL END) AS feb_gsc_avg_position,

        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS feb_ga4_sessions,

        SUM(scroll_events) AS feb_scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    )

    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN SUM(
                CASE
                    WHEN gsc_data_available IS TRUE THEN gsc_clicks
                    ELSE 0
                END
            ) > 0
            THEN 1
            ELSE 0
        END AS march_organic_click_label

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,
    feb.feb_gsc_impressions,
    feb.feb_gsc_clicks,
    feb.feb_gsc_avg_position,
    feb.feb_ga4_sessions,
    feb.feb_scroll_events,
    mar.march_organic_click_label

FROM feb

INNER JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
    AND feb.content_hash_id = mar.content_hash_id

""").df()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (303572, 8)


,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_ga4_sessions,feb_scroll_events,march_organic_click_label
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0,0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0,1
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0,0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,6.0,1.0,1
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,1.0,1


### Distribution check

The February search and engagement signals are highly right-skewed, with many content items having low or zero activity and a smaller number having much larger values. This means the raw distributions have heavy tails, so comparisons should be interpreted carefully rather than assuming a symmetric distribution.

In [29]:
# Inspect distributions, missing values, and upper-tail behavior.

signal_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

distribution_summary = feature_frame[signal_cols].describe(
    percentiles=[0.50, 0.75, 0.95, 0.99]
).T

distribution_summary["missing"] = feature_frame[signal_cols].isna().sum()
distribution_summary["zero"] = (feature_frame[signal_cols] == 0).sum()

distribution_summary[
    [
        "count",
        "mean",
        "50%",
        "75%",
        "95%",
        "99%",
        "max",
        "missing",
        "zero"
    ]
]

,count,mean,50%,75%,95%,99%,max,missing,zero
feb_gsc_impressions,303572.0,585.928976,0.00000,105.000000,2788.000000,10569.290000,203401.0,0,158293
feb_gsc_clicks,303572.0,1.916346,0.00000,0.000000,8.000000,37.000000,3310.0,0,249834
feb_gsc_avg_position,145279.0,12.295121,7.72189,14.036396,40.922091,68.280351,633.0,158293,1527
feb_ga4_sessions,303572.0,1.137328,0.00000,0.000000,3.000000,26.000000,4038.0,0,270225
feb_scroll_events,171861.0,0.236924,0.00000,0.000000,1.000000,4.000000,3037.0,131711,156716


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test results

**Signal #1 — February GSC impressions: CONFIRMED.**  
Content with positive February impressions had a March organic-click rate of 39.88%, compared with 3.30% for content with zero impressions.

**Signal #2 — February GSC clicks: CONFIRMED.**  
Content with positive February clicks had a March organic-click rate of 79.12%, compared with 8.27% for content with zero clicks. This was the strongest of the three tested signals.

**Signal #3 — February GA4 sessions: CONFIRMED.**  
Content with positive February sessions had a March organic-click rate of 45.65%, compared with 17.74% for content with zero sessions.

These tests support the signals as useful directional indicators of future organic activity, but they do not establish causation.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Test whether higher February activity is associated with a higher
# probability of getting an organic click in March.

test_signals = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_ga4_sessions"
]

results = []

for signal in test_signals:
    # Split into zero and positive February activity.
    zero_mask = feature_frame[signal].fillna(0) == 0
    positive_mask = feature_frame[signal].fillna(0) > 0

    zero_rate = feature_frame.loc[
        zero_mask, "march_organic_click_label"
    ].mean()

    positive_rate = feature_frame.loc[
        positive_mask, "march_organic_click_label"
    ].mean()

    difference = positive_rate - zero_rate

    if positive_rate > zero_rate:
        verdict = "CONFIRMED"
    elif positive_rate < zero_rate:
        verdict = "OPPOSITE"
    else:
        verdict = "MIXED"

    results.append({
        "signal": signal,
        "zero_count": int(zero_mask.sum()),
        "positive_count": int(positive_mask.sum()),
        "March click rate: zero": round(zero_rate, 4),
        "March click rate: positive": round(positive_rate, 4),
        "difference": round(difference, 4),
        "verdict": verdict
    })

signal_tests = pd.DataFrame(results)

signal_tests


,signal,zero_count,positive_count,March click rate: zero,March click rate: positive,difference,verdict
0,feb_gsc_impressions,158293,145279,0.0330,0.3988,0.3658,CONFIRMED
1,feb_gsc_clicks,249834,53738,0.0827,0.7912,0.7085,CONFIRMED
2,feb_ga4_sessions,270225,33347,0.1774,0.4565,0.2791,CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: VISIBLE_LOW_CTR

The `VISIBLE_LOW_CTR` flag assumes that content with enough search visibility but very low click-through rate is worth reviewing. I tested the rule using February GSC impressions and clicks, then compared the March organic-click rate for flagged and unflagged content.

The rule is treated as a decision-support signal, not proof that the content is poor or that low CTR causes weaker future performance.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Audit the VISIBLE_LOW_CTR flag:
# high visibility (>= 35 impressions) + very low CTR (< 1%).

flag_test = feature_frame.copy()

# Calculate February CTR safely.
flag_test["feb_ctr"] = np.where(
    flag_test["feb_gsc_impressions"] > 0,
    flag_test["feb_gsc_clicks"] / flag_test["feb_gsc_impressions"],
    np.nan
)

# Recreate the flag rule.
flag_test["visible_low_ctr"] = (
    (flag_test["feb_gsc_impressions"] >= 35) &
    (flag_test["feb_ctr"] < 0.01)
)

# Compare March organic-click rates.
flagged = flag_test["visible_low_ctr"]
unflagged = ~flagged

flagged_rate = flag_test.loc[
    flagged, "march_organic_click_label"
].mean()

unflagged_rate = flag_test.loc[
    unflagged, "march_organic_click_label"
].mean()

difference = flagged_rate - unflagged_rate

print("Flagged rows:", int(flagged.sum()))
print("Unflagged rows:", int(unflagged.sum()))
print(f"March click rate - flagged:   {flagged_rate:.4f}")
print(f"March click rate - unflagged: {unflagged_rate:.4f}")
print(f"Difference (flagged - unflagged): {difference:.4f}")

if flagged_rate < unflagged_rate:
    verdict = "CONFIRMED"
elif flagged_rate > unflagged_rate:
    verdict = "OPPOSITE"
else:
    verdict = "MIXED"

print("Verdict:", verdict)

Flagged rows: 88674
Unflagged rows: 214898
March click rate - flagged:   0.5616
March click rate - unflagged: 0.0622
Difference (flagged - unflagged): 0.4994
Verdict: OPPOSITE


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The February signal audit supports using search visibility, search clicks, and GA4 sessions as directional indicators when prioritizing content for human review, with February GSC clicks showing the strongest association with March organic clicks. Because the signals are highly zero-heavy and right-skewed, the content team should use them for prioritization and review rather than treating any single raw value as proof of content quality or future performance. These relationships are observational and should not be interpreted as causal.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Summarize the strongest directional signal from the three tests.

strongest = signal_tests.loc[
    signal_tests["difference"].idxmax()
]

print("Strongest tested signal:", strongest["signal"])
print(
    "March click-rate difference:",
    f'{strongest["difference"]:.4f}'
)
print(
    "Verdict:",
    strongest["verdict"]
)

print("\nPractical takeaway:")
print(
    "Use the audited signals for directional content-review prioritization, "
    "not as causal or automatic quality judgments."
)


Strongest tested signal: feb_gsc_clicks
March click-rate difference: 0.7085
Verdict: CONFIRMED

Practical takeaway:
Use the audited signals for directional content-review prioritization, not as causal or automatic quality judgments.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.